In [ ]:
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 xformers --index-url https://download.pytorch.org/whl/cu124

!git clone https://github.com/resemble-ai/chatterbox.git
%cd chatterbox
!pip install -e .
%cd ..

In [1]:
%cd chatterbox

/home/aali2/TTS_low_resource/chatterbox


In [ ]:
import torchaudio as ta
from src.chatterbox.tts import ChatterboxTTS

model = ChatterboxTTS.from_pretrained(device="cuda")


In [2]:
import shutil
from pathlib import Path

def copy_dereference(src_dir: str, dst_dir: str):
    src = Path(src_dir)
    dst = Path(dst_dir)

    # Ensure destination is a clean directory
    if dst.exists() and dst.is_file():
        dst.unlink()  # remove if it's accidentally a file
    dst.mkdir(parents=True, exist_ok=True)

    for item in src.iterdir():
        if item.name.startswith('.'):
            continue

        if item.is_symlink() or item.is_file():
            try:
                real_src = item.resolve()
                if real_src.is_file():
                    # Step 1: Copy into dst (uses blob hash as filename)
                    copied_path = shutil.copy2(real_src, dst)
                    # Step 2: Rename to the symlink's original name
                    new_path = dst / item.name
                    # If file already exists, overwrite it
                    if new_path.exists():
                        new_path.unlink()
                    Path(copied_path).rename(new_path)
                    print(f"✅ Copied as: {item.name}")
                else:
                    print(f"⚠️ Skipping non-file: {item.name}")
            except Exception as e:
                print(f"❌ Failed on {item.name}: {e}")
        else:
            print(f"📁 Skipping directory: {item.name}")

# Usage
copy_dereference(
    "/home/aali2/.cache/huggingface/hub/models--ResembleAI--chatterbox/snapshots/05e904af2b5c7f8e482687a9d7336c5c824467d9",
    "/home/aali2/TTS_low_resource/chatterbox/temp_model"
)

✅ Copied as: ve.safetensors
✅ Copied as: t3_cfg.safetensors
✅ Copied as: s3gen.safetensors
✅ Copied as: tokenizer.json
✅ Copied as: conds.pt


In [3]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from datasets import load_dataset
import os
import tempfile

# -------------------------------
# 1. Load the dataset
# -------------------------------

print("📥 Loading the dataset")

try:
    dataset = load_dataset("Vikhrmodels/ToneSpeak", split="train")
except Exception as e:
    raise RuntimeError(f"Failed to load 'ToneSpeak' dataset. Make sure you're logged in to Hugging Face. Error: {e}")

print(f"✅ Loaded {len(dataset)} examples.")

# Extract text 

def extract_text(example, name="text"):
    return {"text": example[name]}

# Apply mapping to extract text
print("🧹 Extracting text with tashkeel...")
texts = dataset.map(extract_text, num_proc=4, remove_columns=dataset.column_names)


# Save to temporary file for tokenizer training
temp_dir = tempfile.mkdtemp()
temp_file = os.path.join(temp_dir, "ru.txt")

print("🧹 Saving Russian text to file...")

with open(temp_file, "w", encoding="utf-8") as f:
    for i in range(len(texts)):
        example = texts[i]  # This ensures we get a dict
        text = example["text"].strip()
        if text:
            f.write(text + "\n")
        if i % 5000 == 0:
            print(f"📝 Written {i} lines")

print(f"✅ Text saved to {temp_file}")

# -------------------------------
# 2. Initialize BPE Tokenizer
# -------------------------------

tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# -------------------------------
# 3. Configure BPE Trainer (vocab_size = 2000)
# -------------------------------

trainer = trainers.BpeTrainer(
    vocab_size=300,
    min_frequency=1000,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
    show_progress=True,
)

# -------------------------------
# 4. Train the Tokenizer
# -------------------------------

print("🚀 Training BPE tokenizer on Russian text ")

tokenizer.train(files=[temp_file], trainer=trainer)

# -------------------------------
# 5. Save the Tokenizer
# -------------------------------

output_dir = "./ru_bpe_2000"
os.makedirs(output_dir, exist_ok=True)
tokenizer.save(f"{output_dir}/tokenizer.json")
tokenizer.model.save(output_dir)  # Saves vocab.json and merges.txt

print(f"✅ Tokenizer trained and saved to '{output_dir}'")
print(f"📊 Final vocabulary size: {tokenizer.get_vocab_size()}")

# -------------------------------
# 7. Cleanup
# -------------------------------

import shutil
shutil.rmtree(temp_dir)
print(f"🧹 Temporary data directory removed.")

/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


📥 Loading the dataset
✅ Loaded 6298 examples.
🧹 Extracting text with tashkeel...
🧹 Saving Russian text to file...
📝 Written 0 lines
📝 Written 5000 lines
✅ Text saved to /tmp/tmp3m_jukto/ru.txt
🚀 Training BPE tokenizer on Russian text 



✅ Tokenizer trained and saved to './ru_bpe_2000'
📊 Final vocabulary size: 224
🧹 Temporary data directory removed.


In [4]:
from tokenizers import Tokenizer

# Load the saved tokenizer
tokenizer_path = "./ru_bpe_2000/tokenizer.json"
tokenizer = Tokenizer.from_file(tokenizer_path)

print("✅ Tokenizer loaded successfully!")
print(f"📊 Vocabulary size: {tokenizer.get_vocab_size()}")
ids = tokenizer.encode("привет").ids
tokenizer.decode(ids), ids

✅ Tokenizer loaded successfully!
📊 Vocabulary size: 224


('при вет', [116, 115])

In [5]:
from src.chatterbox.models.tokenizers.tokenizer import EnTokenizer
tokenizer_orig = EnTokenizer("./temp_model/tokenizer.json")
import json
with open("./temp_model/tokenizer.json", "r") as f:
    state = json.load(f)
state2 = json.loads(tokenizer.__getstate__())
state2.keys()  


vocab = state["model"]["vocab"]  # {token: index}
vocab2 = state2["model"]["vocab"]  # {token: index}
for key in vocab2.keys():
    vocab2[key] += 704
merges1 = state["model"]["merges"]  # {token: index}
merges2 = state2["model"]["merges"]  # {token: index}
def merge_bpe_merges(merges1, merges2):
    """
    Merge two BPE merge lists:
    - merges1: list of strings like "a b"
    - merges2: list of lists like ['a', 'b']
    Returns: list of strings in format "a b", no duplicates, order preserved.
    """
    seen = set()
    merged = []

    # Add merges1
    for merge in merges1:
        if merge not in seen:
            seen.add(merge)
            merged.append(merge)

    # Add merges2 (convert ['a', 'b'] -> "a b")
    for pair in merges2:
        merge_str = " ".join(pair)
        if merge_str not in seen:
            seen.add(merge_str)
            merged.append(merge_str)

    return merged

# Rebuild vocab with new indices
new_vocab = {}
next_id = 0
# Add all tokens from vocab1 first
for token in vocab:
    if token not in new_vocab:
        new_vocab[token] = next_id
        next_id += 1
# Add tokens from vocab2, skipping duplicates
for token in vocab2:
    if token not in new_vocab:
        new_vocab[token] = next_id
        next_id += 1
state["model"]["vocab"] = new_vocab

state["model"]["merges"] = merge_bpe_merges(merges1, merges2)

with open("en_ru_tokenizer.json", "w") as f:
    json.dump(state,f)

/opt/conda/lib/python3.11/site-packages/perth/perth_net/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [6]:
tokenizer_new = EnTokenizer("./en_ru_tokenizer.json")

In [7]:
def transliterate_ru_to_latin(text: str) -> str:
    """
    Transliterates Russian Cyrillic text to Latin using GOST 7.79-2000 (System B).
    
    Args:
        text (str): Input string in Russian (Cyrillic).
    
    Returns:
        str: Transliterated string in Latin characters.
    """
    mapping = {
        'а': 'a', 'б': 'b', 'в': 'v', 'г': 'g', 'д': 'd',
        'е': 'e', 'ё': 'yo', 'ж': 'zh', 'з': 'z', 'и': 'i',
        'й': 'j', 'к': 'k', 'л': 'l', 'м': 'm', 'н': 'n',
        'о': 'o', 'п': 'p', 'р': 'r', 'с': 's', 'т': 't',
        'у': 'u', 'ф': 'f', 'х': 'kh', 'ц': 'ts', 'ч': 'ch',
        'ш': 'sh', 'щ': 'shh', 'ъ': '', 'ы': 'y', 'ь': '',
        'э': 'e', 'ю': 'yu', 'я': 'ya',
        'А': 'A', 'Б': 'B', 'В': 'V', 'Г': 'G', 'Д': 'D',
        'Е': 'E', 'Ё': 'Yo', 'Ж': 'Zh', 'З': 'Z', 'И': 'I',
        'Й': 'J', 'К': 'K', 'Л': 'L', 'М': 'M', 'Н': 'N',
        'О': 'O', 'П': 'P', 'Р': 'R', 'С': 'S', 'Т': 'T',
        'У': 'U', 'Ф': 'F', 'Х': 'Kh', 'Ц': 'Ts', 'Ч': 'Ch',
        'Ш': 'Sh', 'Щ': 'Shh', 'Ъ': '', 'Ы': 'Y', 'Ь': '',
        'Э': 'E', 'Ю': 'Yu', 'Я': 'Ya'
    }

    return ''.join(mapping.get(char, char) for char in text)

In [8]:
# from safetensors import safe_open
# from safetensors.torch import save_file
# def load_safetensors_to_torch(file_path: str, device: str = "cpu") -> dict:
#     """Load a safetensors file into a PyTorch state dict."""
#     print(f"Loading safetensors file: {file_path}")
#     state_dict = {}
#     with safe_open(file_path, framework="pt", device=device) as f:
#         for key in f.keys():
#             tensor = f.get_tensor(key)
#             state_dict[key] = tensor
#     print(f"Successfully loaded {len(state_dict)} tensors.")
#     return state_dict
# chk = load_safetensors_to_torch("./temp_model/t3_cfg.safetensors")
# import torch
# embed = chk["text_emb.weight"]
# embed2 = chk["text_head.weight"]
# new_vocab_size = max(idx for idx in tokenizer_new.tokenizer.get_vocab().values()) + 1
# new_embed = torch.zeros(new_vocab_size, 1024)
# new_embed2 = torch.zeros(new_vocab_size, 1024)


# new_embed[:704] = embed[:704].clone()
# new_embed2[:704] = embed2[:704].clone()

# tot = 0
# # Iterate by actual token ID (0 to vocab_size-1)
# for token_id in range(new_vocab_size):
#     if token_id < 704:
#         continue
        
#     # Get token string using token ID (correct method)
#     ru = tokenizer_new.decode([token_id])
#     word = transliterate_ru_to_latin(ru)
    
#     try:
#         ids = tokenizer_orig.encode(word)
#         if not ids:  # Handle empty encoding
#             raise ValueError("Empty encoding result")
            
#         # Initialize accumulators
#         sume = torch.zeros(1024)
#         sume2 = torch.zeros(1024)
        
#         # Sum embeddings for all subtokens
#         for i in ids:
#             sume += embed[i]
#             sume2 += embed2[i]
            
#         # Store averaged embedding
#         new_embed[token_id] = sume / len(ids)
#         new_embed2[token_id] = sume2 / len(ids)
        
#     except Exception as e:
#         print(f"Failed with token {word}, {ru}: {str(e)}")
#         tot += 1

# # Update checkpoint with new embeddings
# chk["text_emb.weight"] = new_embed
# chk["text_head.weight"] = new_embed2
# save_file(chk, "./temp_model/t3_cfg.safetensors")

In [9]:
import torch
import numpy as np
from safetensors import safe_open
from safetensors.torch import save_file

def lerp(t, v0, v1):
    return (1 - t) * v0 + t * v1

def slerp(t, v0, v1, DOT_THRESHOLD=0.9995):
    """Spherical linear interpolation between two 1D tensors (vectors)."""
    # Ensure float32 for stability
    v0 = v0.to(dtype=torch.float32)
    v1 = v1.to(dtype=torch.float32)

    norm_v0 = torch.norm(v0)
    norm_v1 = torch.norm(v1)

    epsilon = 1e-10
    if norm_v0 < epsilon or norm_v1 < epsilon:
        # Fall back to lerp if either vector is near zero
        return lerp(t, v0, v1)

    # Normalize
    u0 = v0 / norm_v0
    u1 = v1 / norm_v1

    dot = torch.sum(u0 * u1)
    dot = torch.clamp(dot, -1.0, 1.0)  # Clamp for numerical safety

    if torch.abs(dot) > DOT_THRESHOLD:
        # Vectors are nearly parallel → use lerp
        return lerp(t, v0, v1)

    theta_0 = torch.acos(dot)
    sin_theta_0 = torch.sin(theta_0)

    if sin_theta_0 < epsilon:
        return lerp(t, v0, v1)

    theta_t = theta_0 * t
    sin_theta_t = torch.sin(theta_t)

    s0 = torch.sin(theta_0 - theta_t) / sin_theta_0
    s1 = sin_theta_t / sin_theta_0

    return s0 * v0 + s1 * v1
def slerp_multiple(vectors):
    """
    Slerp a list of 1D tensors (same shape) into a single representative vector.
    Uses recursive pairwise slerp with equal weighting.
    """
    if not vectors:
        raise ValueError("Empty vector list")
    if len(vectors) == 1:
        return vectors[0].clone()

    result = vectors[0]
    for i in range(1, len(vectors)):
        # Weight: interpolate current result with next vector
        # At step i (0-indexed), we have i+1 vectors total so far.
        # To give equal weight, use t = 1/(i+1)
        t = 1.0 / (i + 1)
        result = slerp(t, result, vectors[i])
    return result
# Load model

def slerp_weighted_sequence(vectors, weights=None):
    """
    Combine vectors in order with optional weights (e.g., positional decay).
    If no weights, uses uniform weighting via recursive slerp.
    """
    if weights is None:
        weights = torch.ones(len(vectors))
    else:
        weights = torch.tensor(weights, dtype=torch.float32)
    
    # Normalize weights
    weights = weights / weights.sum()

    result = vectors[0]
    cum_weight = weights[0]
    for i in range(1, len(vectors)):
        next_weight = weights[i]
        t = next_weight / (cum_weight + next_weight)  # interpolation ratio
        result = slerp(t, result, vectors[i])
        cum_weight += next_weight
    return result

def descending_linear_weights(n):
    """
    Generate normalized descending linear weights for n elements.
    Example: n=4 → [4, 3, 2, 1] → normalized to [0.4, 0.3, 0.2, 0.1]
    """
    if n <= 0:
        raise ValueError("n must be positive")
    weights = torch.arange(n, 0, -1, dtype=torch.float32)  # [n, n-1, ..., 1]
    return weights / weights.sum()  # normalize to sum=1
    
def load_safetensors_to_torch(file_path: str, device: str = "cpu") -> dict:
    print(f"Loading safetensors file: {file_path}")
    state_dict = {}
    with safe_open(file_path, framework="pt", device=device) as f:
        for key in f.keys():
            state_dict[key] = f.get_tensor(key)
    print(f"Successfully loaded {len(state_dict)} tensors.")
    return state_dict

chk = load_safetensors_to_torch("./temp_model/t3_cfg.safetensors")
embed = chk["text_emb.weight"]      # [old_vocab, dim]
embed2 = chk["text_head.weight"]    # [old_vocab, dim]

new_vocab_size = max(idx for idx in tokenizer_new.tokenizer.get_vocab().values()) + 1
dim = embed.shape[1]  # e.g., 1024

new_embed = torch.zeros(new_vocab_size, dim)
new_embed2 = torch.zeros(new_vocab_size, dim)

# Copy original tokens (assumed 0–703 are shared)
new_embed[:704] = embed[:704].clone()
new_embed2[:704] = embed2[:704].clone()

tot = 0
for token_id in range(704, new_vocab_size):
    ru = tokenizer_new.decode([token_id])
    word = transliterate_ru_to_latin(ru)

    try:
        ids = tokenizer_orig.encode(word)
        if not ids:
            raise ValueError("Empty encoding")
        n = len(ids)
        # Collect all subtoken embeddings
        emb_list = [embed[i] for i in ids]
        emb2_list = [embed2[i] for i in ids]
        weights = descending_linear_weights(n)
        # Use slerp to merge
        merged_emb = slerp_weighted_sequence(emb_list, weights)
        merged_emb2 = slerp_weighted_sequence(emb2_list, weights)

        new_embed[token_id] = merged_emb
        new_embed2[token_id] = merged_emb2

    except Exception as e:
        print(f"Failed with token {word} ({ru}): {str(e)}")
        tot += 1

print(f"Failed to map {tot} tokens.")

# Update checkpoint
chk["text_emb.weight"] = new_embed
chk["text_head.weight"] = new_embed2

# Save
save_file(chk, "./temp_model/t3_cfg.safetensors")

Loading safetensors file: ./temp_model/t3_cfg.safetensors
Successfully loaded 292 tensors.


/tmp/ipykernel_2715204/3457862269.py:75: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  weights = torch.tensor(weights, dtype=torch.float32)


Failed with token  (ъ): Empty encoding
Failed with token  (ь): Empty encoding
Failed to map 2 tokens.


In [10]:
tokenizer_new.encode("привет")

[803, 802]

In [11]:
new_embed[802]

tensor([-0.0375,  0.0014,  0.0030,  ...,  0.0028,  0.0350, -0.0376])

In [12]:
tokenizer_orig.encode("privet")

[29, 101, 76, 33]

In [13]:
(embed[76] + embed[33]) /2

tensor([-0.0163,  0.0024,  0.0045,  ...,  0.0076,  0.0256, -0.0248])

In [14]:
%cp en_ru_tokenizer.json -d ./temp_model/tokenizer.json

In [15]:
new_vocab_size

911

In [16]:
%%writefile ./src/chatterbox/models/t3/modules/t3_config.py
from ..llama_configs import LLAMA_CONFIGS


class T3Config:
    def __init__(self, text_tokens_dict_size=911):
        self.start_text_token = 255
        self.stop_text_token = 0
        self.text_tokens_dict_size = text_tokens_dict_size
        self.max_text_tokens = 2048

        self.start_speech_token = 6561
        self.stop_speech_token = 6562
        self.speech_tokens_dict_size = 8194
        self.max_speech_tokens = 4096

        self.llama_config_name = "Llama_520M"
        self.input_pos_emb = "learned"
        self.speech_cond_prompt_len = 150

        self.encoder_type = "voice_encoder"
        self.speaker_embed_size = 256
        self.use_perceiver_resampler = True
        self.emotion_adv = True

    @property
    def n_channels(self):
        return LLAMA_CONFIGS[self.llama_config_name]["hidden_size"]
    
    @property
    def is_multilingual(self):
        return self.text_tokens_dict_size == 2454

    @classmethod
    def english_only(cls):
        """Create configuration for English-only TTS model."""
        return cls(text_tokens_dict_size=911)
    
    @classmethod 
    def multilingual(cls):
        """Create configuration for multilingual TTS model."""
        return cls(text_tokens_dict_size=2454)


Overwriting ./src/chatterbox/models/t3/modules/t3_config.py


## restart notebook

In [1]:
%cd chatterbox

/home/aali2/TTS_low_resource/chatterbox


In [2]:
import torchaudio as ta
from src.chatterbox.tts import ChatterboxTTS

model = ChatterboxTTS.from_local(ckpt_dir="./temp_model",device="cuda")


/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/opt/conda/lib/python3.11/site-packages/perth/perth_net/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/opt/conda/lib/python3.11/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", depr

loaded PerthNet (Implicit) at step 250,000


In [6]:
text = "меня зовут Мага"
wav = model.generate(text)
ta.save("test-ru.wav", wav, model.sr)

Sampling:   4%|▍         | 41/1000 [00:00<00:21, 43.94it/s]
